# Research Paper Analyzer using LLMs

This project extracts key information from research papers (PDF format) and uses AI models to simplify and translate the content.

---

## Features
- Extracts **Title** and **Abstract** from research papers
- Uses **Hugging Face LLM** to simplify complex research text
- Translates output into **user-selected regional language** using Sarvam AI
- Makes research papers easier to understand for non-experts

---

## Pipeline

PDF → Text Extraction → Abstract Detection → LLM Simplification → Translation (Sarvam AI)

In [72]:
!pip install pymupdf transformers huggingface_hub

In [73]:
import fitz
import re
import requests
from transformers import pipeline

In [74]:
from google.colab import files
uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]

Saving 1706.03762v7.pdf to 1706.03762v7.pdf
Saving 1810.04805v2.pdf to 1810.04805v2.pdf


### Extract Text

In [194]:
def extract_title_and_text(pdf_path):
    doc = fitz.open(pdf_path)

    page = doc[0]
    blocks = page.get_text("dict")["blocks"]

    candidates = []

    for block in blocks:
        if "lines" in block:
            for line in block["lines"]:
                line_text = ""
                max_size = 0

                for span in line["spans"]:
                    line_text += span["text"]
                    max_size = max(max_size, span["size"])

                candidates.append((max_size, line_text.strip()))
    candidates = sorted(candidates, key=lambda x: x[0], reverse=True)
    for size, text in candidates:
        if (
            len(text) > 10 and
            not any(x in text.lower() for x in ["arxiv", "http", "doi", "license", "permission"]) and
            text.isascii()
        ):
            title = text
            break
    full_text = ""
    for page in doc:
        full_text += page.get_text()

    return title, full_text

### Abstract

In [195]:
def extract_abstract(text):
    text_lower = text.lower()

    start = text_lower.find("abstract")
    end = text_lower.find("introduction")

    if start != -1 and end != -1:
        return text[start:end]
    else:
        return "Abstract not found"

abstract = extract_abstract(text)

### Print Extracted Texts

In [196]:
title, text = extract_title_and_text(pdf_path)
print("TITLE:\n", title,"\n")
print(abstract[::])

TITLE:
 Attention Is All You Need 

Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to
be superior in quality while being more parallelizable and requiring significantly
less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English-
to-German translation task, improving over the existing best results, including
ensembles, by over 2 BLEU. On the WMT 2014 English-to-French translation task,
our model establishes a new single-model state-of-the-art BLEU score of 41.8 after
training for 3.5 days on eight GPUs, a small fraction of the training costs of the
best models

In [201]:
def clean_abstract_text(text):
    stop_words = ["∗", "†", "‡", "arxiv", "conference"]

    for word in stop_words:
        idx = text.lower().find(word)
        if idx != -1:
            text = text[:idx]

    return text.strip()

In [203]:
def chunk_text(text):
    sentences = text.split(". ")
    chunks = []
    for i in range(0, len(sentences), 4):
        chunk = ". ".join(sentences[i:i+4])
        if len(chunk.strip()) > 20:
            chunks.append(chunk.strip())

    return chunks

In [205]:
clean_abstract = clean_abstract.replace("Abstract", "")
chunks = chunk_text(clean_abstract)
print("\nGENERATED CHUNKS:\n")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ---")
    print(chunk)
    print()


GENERATED CHUNKS:

--- Chunk 1 ---
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to
be superior in quality while being more parallelizable and requiring significantly
less time to train

--- Chunk 2 ---
Our model achieves 28.4 BLEU on the WMT 2014 English-
to-German translation task, improving over the existing best results, including
ensembles, by over 2 BLEU. On the WMT 2014 English-to-French translation task,
our model establishes a new single-model state-of-the-art BLEU score of 41.8 after
training for 3.5 days on eight GPUs, a small fraction of the training costs of the
best

### HuggingFace Summarization

In [206]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
def summarize(text):
    prompt = f"""
Summarize this abstract without loosing the meaning of the text.

{text}
"""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    outputs = model.generate(
        inputs["input_ids"],
        max_length=200,
        num_beams=4,
        early_stopping=True
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [207]:
def summarize_chunks(chunks):
    bullet_points = []
    for chunk in chunks:
        summary = summarize(chunk)
        summary = summary.replace("\n", " ").strip()
        bullet_points.append(f"- {summary}")
    return bullet_points

In [211]:
bullet_points = summarize_chunks(chunks)
print("\nCHUNK-WISE SUMMARY:\n")

for point in bullet_points:
    print(point)


CHUNK-WISE SUMMARY:

- We propose recurrent and convolutional neural networks for machine translation tasks. We propose a transformer based solely on attention mechanisms, dispensing with recurrence and convolutions.
- We present a state-of-the-art single-model state-of-the-art BLEU score of 41.8 after training for 3.5 days on eight GPUs, a small fraction of the training costs of the best models from the literature. We show that the Transformer generalizes well to other tasks by applying it successfully to English constituency parsing both with large and limited training data
- We are proud to say that we've had a number of significant contributions to this work. We'd like to congratulate Jakob, Ashish, Noam and Niki on their contributions.
- tensor2tensor is a neural information processing system designed and implemented by Lukasz Lukasz and Aidan Lukasz.


### User chooses language

In [212]:
lang_map = {
    "hindi": "hi-IN",
    "english": "en-IN",
    "tamil": "ta-IN",
    "telugu": "te-IN",
    "bengali": "bn-IN",
    "gujarati": "gu-IN",
    "marathi": "mr-IN",
    "kannada": "kn-IN",
    "malayalam": "ml-IN",
    "punjabi": "pa-IN",
    "urdu": "ur-IN"
}

In [213]:
user_lang = input("Enter language (Hindi, Tamil, Bengali, etc): ").lower().strip()

lang = lang_map.get(user_lang)

if not lang:
    print("Language not supported, defaulting to Hindi")
    lang = "hi-IN"

Enter language (Hindi, Tamil, Bengali, etc): hindi


### Sarvam AI Translation

In [214]:
!pip install -U sarvamai

In [215]:
import os
from getpass import getpass

os.environ["SARVAM_API_KEY"] = "YOUR_SARVAM_API_KEY" #Please add your sarvam ai api key here

In [130]:
from sarvamai import SarvamAI
import os

client = SarvamAI(
    api_subscription_key=os.environ["SARVAM_API_KEY"]
)
def split_text(text, max_len=900):
    chunks = []
    while len(text) > max_len:
        split_idx = text.rfind(" ", 0, max_len)
        if split_idx == -1:
            split_idx = max_len
        chunks.append(text[:split_idx])
        text = text[split_idx:]
    chunks.append(text)
    return chunks
def translate_sarvam(text, target_lang):
    chunks = split_text(text)
    translated_chunks = []

    for chunk in chunks:
        try:
            response = client.text.translate(
                input=chunk,
                source_language_code="en-IN",
                target_language_code=target_lang,
                speaker_gender="Male"
            )
            translated_chunks.append(response.translated_text)

        except Exception as e:
            return f"Sarvam Error: {str(e)}"

    return " ".join(translated_chunks)

In [216]:
def translate_bullets(bullet_points, target_lang):
    translated_points = []

    for point in bullet_points:
        clean_point = point.lstrip("- ").strip()
        translated = translate_sarvam(clean_point, target_lang)
        translated_points.append(f"- {translated}")

    return translated_points

In [217]:
translated_points = translate_bullets(bullet_points, lang)
print("\nTRANSLATED SUMMARY:\n")

for t in translated_points:
    print(t)


TRANSLATED SUMMARY:

- हम मशीन अनुवाद कार्यों के लिए आवर्ती और कन्वोलुशनल तंत्रिका तंत्र का प्रस्ताव करते हैं। हम पुनरावृत्ति और आवर्तन को छोड़कर, केवल ध्यान तंत्र पर आधारित एक ट्रांसफार्मर का प्रस्ताव करते हैं।
- हम आठ जी.पी.यू. पर 3.5 दिनों के प्रशिक्षण के बाद 41.8 का एक अत्याधुनिक एकल-मॉडल अत्याधुनिक बी.एल.ई.यू. स्कोर प्रस्तुत करते हैं, जो साहित्य के सर्वश्रेष्ठ मॉडलों के प्रशिक्षण लागत का एक छोटा अंश है। हम दिखाते हैं कि ट्रांसफार्मर अन्य कार्यों के लिए अच्छी तरह से सामान्यीकृत होता है, क्योंकि इसे बड़े और सीमित प्रशिक्षण डेटा के साथ अंग्रेजी निर्वाचन क्षेत्र पार्सिंग पर सफलतापूर्वक लागू किया जाता है।
- हम यह कहते हुए गौरान्वित हैं कि इस कार्य में हमारे द्वारा किए गए कई महत्वपूर्ण योगदान हैं। हम जैकब, आशीष, नोआम और निकी को उनके योगदान के लिए बधाई देना चाहते हैं।
- टेन्सर2टेन्सर एक तंत्रिका सूचना प्रसंस्करण प्रणाली है जिसे लुकाश लुकाश और एडन लुकाश द्वारा डिज़ाइन और कार्यान्वित किया गया है।


In [223]:
translated_title = translate_sarvam(title, lang)
print("\nTITLE (ENGLISH):\n")
print(title)

print("\nTITLE (TRANSLATED):\n")
print(translated_title)


TITLE (ENGLISH):

Attention Is All You Need

TITLE (TRANSLATED):

बस ध्यान देना होगा।


In [224]:
print("\n" + "="*60)
print("RESEARCH PAPER ANALYSIS REPORT")
print("="*60)

# TITLE
print("\nTITLE (ENGLISH):\n")
print(title)

print("\nTITLE (TRANSLATED):\n")
print(translated_title)

# ABSTRACT
print("\nABSTRACT (EXTRACTED):\n")
print(abstract[:800] + "...")

# ENGLISH SUMMARY
print("\nENGLISH SUMMARY:\n")
for b in bullet_points:
    print(b)

# TRANSLATED SUMMARY
print("\nTRANSLATED SUMMARY:\n")
for t in translated_points:
    print(t)

print("\n" + "="*60)
print("END OF REPORT")
print("="*60)


RESEARCH PAPER ANALYSIS REPORT

TITLE (ENGLISH):

Attention Is All You Need

TITLE (TRANSLATED):

बस ध्यान देना होगा।

ABSTRACT (EXTRACTED):

Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to
be superior in quality while being more parallelizable and requiring significantly
less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English-
to-German translation task, improving over the existing best results, including
ensembles, by over 2 BLEU. On the WMT 2014 English-to-French translation task,
our model est...

ENGLISH SUMMARY:

- We propose recurrent and convolu